# 🧠 RNN Worksheet: Urdu → English Machine Translation
### Applied Deep Learning | Sequence-to-Sequence with Vanilla RNN

---

**Instructions:**
- Every cell marked `# ✏️ YOUR CODE HERE` requires you to write code.
- Do **not** modify cells marked `# ✅ PROVIDED`.
- Run cells **in order** from top to bottom.
- Read comments and docstrings carefully before writing.

**What you will build today:**
```
 Urdu sentence  →  [Encoder RNN]  →  hidden state  →  [Decoder RNN]  →  English sentence
```

**Framework:** PyTorch &nbsp;|&nbsp; **Model:** Vanilla RNN (no GRU, no LSTM)


---
## Section 0 — Setup

In [ ]:
# ✅ PROVIDED
!pip install torch sacrebleu datasets --quiet

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import random, time, re
from collections import Counter
import sacrebleu
import matplotlib.pyplot as plt

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
random.seed(42)
torch.manual_seed(42)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 2.9 MB/s eta 0:00:00
Device: cpu


---
## Section 1 — Dataset

We use a real **Urdu–English** sentence-pair dataset (Helsinki-NLP, HuggingFace).  
A 50-sentence fallback is included so the worksheet works offline too.

In [ ]:
# ✅ PROVIDED
FALLBACK_PAIRS = [
    ('میں ٹھیک ہوں', 'I am fine'),
    ('آپ کا نام کیا ہے', 'What is your name'),
    ('میرا نام احمد ہے', 'My name is Ahmed'),
    ('آج موسم اچھا ہے', 'The weather is nice today'),
    ('مجھے پانی چاہیے', 'I need water'),
    ('کتاب میز پر ہے', 'The book is on the table'),
    ('وہ اسکول جاتا ہے', 'He goes to school'),
    ('میں کھانا کھاتا ہوں', 'I eat food'),
    ('بچے کھیل رہے ہیں', 'The children are playing'),
    ('رات کو تارے چمکتے ہیں', 'Stars shine at night'),
    ('میں پاکستان میں رہتا ہوں', 'I live in Pakistan'),
    ('وہ ڈاکٹر ہے', 'She is a doctor'),
    ('مجھے اردو پسند ہے', 'I like Urdu'),
    ('آج جمعہ ہے', 'Today is Friday'),
    ('دروازہ بند کرو', 'Close the door'),
    ('مجھے نیند آ رہی ہے', 'I am feeling sleepy'),
    ('وہ بہت محنتی ہے', 'He is very hardworking'),
    ('میں کام کر رہا ہوں', 'I am working'),
    ('آپ کہاں سے ہیں', 'Where are you from'),
    ('یہ میری کتاب ہے', 'This is my book'),
    ('بازار قریب ہے', 'The market is nearby'),
    ('ٹرین آ گئی', 'The train has arrived'),
    ('میں اردو پڑھتا ہوں', 'I read Urdu'),
    ('وہ گھر میں ہے', 'She is at home'),
    ('بارش ہو رہی ہے', 'It is raining'),
    ('میں خوش ہوں', 'I am happy'),
    ('کھانا تیار ہے', 'The food is ready'),
    ('یہ شہر بڑا ہے', 'This city is big'),
    ('وہ پڑھ رہی ہے', 'She is studying'),
    ('مجھے بھوک لگی ہے', 'I am hungry'),
    ('سڑک خطرناک ہے', 'The road is dangerous'),
    ('کل چھٹی ہے', 'Tomorrow is a holiday'),
    ('بچہ رو رہا ہے', 'The child is crying'),
    ('میں بازار جاؤں گا', 'I will go to the market'),
    ('کمرہ صاف ہے', 'The room is clean'),
    ('وہ خط لکھ رہا ہے', 'He is writing a letter'),
    ('آج گرمی ہے', 'It is hot today'),
    ('پرندے گا رہے ہیں', 'The birds are singing'),
    ('میں سونا چاہتا ہوں', 'I want to sleep'),
    ('وہ اچھا انسان ہے', 'He is a good person'),
    ('میں اپنے گھر جاتا ہوں', 'I go to my house'),
    ('آپ کیسے ہیں', 'How are you'),
    ('وقت پر آؤ', 'Come on time'),
    ('گاڑی تیز چل رہی ہے', 'The car is running fast'),
    ('دودھ ٹھنڈا ہے', 'The milk is cold'),
    ('کتا بھونک رہا ہے', 'The dog is barking'),
    ('یہ سوال مشکل ہے', 'This question is difficult'),
    ('وہ سو رہی ہے', 'She is sleeping'),
    ('میں تھکا ہوا ہوں', 'I am tired'),
    ('خدا حافظ', 'Goodbye'),
]

def load_pairs(max_samples=3000):
    try:
        from datasets import load_dataset
        print('Loading from HuggingFace...')
        ds = load_dataset('Helsinki-NLP/opus-100', 'en-ur', split='train', trust_remote_code=True)
        pairs = []
        for item in ds:
            ur = item['translation']['ur'].strip()
            en = item['translation']['en'].strip()
            if ur and en:
                pairs.append((ur, en))
            if len(pairs) >= max_samples:
                break
        print(f'Loaded {len(pairs)} pairs.')
        return pairs
    except Exception as e:
        print(f'Using fallback dataset ({e})')
        return FALLBACK_PAIRS

all_pairs = load_pairs()

print('\nSample pairs:')
for ur, en in random.sample(all_pairs, 3):
    print(f'  UR: {ur}  →  EN: {en}')

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'Helsinki-NLP/opus-100' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'Helsinki-NLP/opus-100' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Loading from HuggingFace...


en-ur/test-00000-of-00001.parquet:   0%|          | 0.00/301k [00:00<?, ?B/s]

en-ur/train-00000-of-00001.parquet:   0%|          | 0.00/148M [00:00<?, ?B/s]

en-ur/validation-00000-of-00001.parquet:   0%|          | 0.00/296k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/753913 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Loaded 3000 pairs.

Sample pairs:
  UR: اور یہ دنیاوی زندگی تو محض کھیل تماشہ ہے اور حقیقی زندگی تو آخرت والی ہے۔ کاش لوگوں کو اس (حقیقت) کا علم ہوتا۔  →  EN: And this life of the world is only amusement and play! Verily, the home of the Hereafter, that is the life indeed (i.e. the eternal life that will never end), if they but knew
  UR: یہودی کہتے ہیں: یہودی ہو تو راہ راست پاؤ گے عیسائی کہتے ہیں: عیسائی ہو، تو ہدایت ملے گی اِن سے کہو: "نہیں، بلکہ سب کو چھوڑ کر ابراہیمؑ کا طریقہ اور ابراہیمؑ مشر کو ں میں سے نہ تھا"  →  EN: And they say, "Be Jews or Christians, then you will be guided." Say (to them, O Muhammad Peace be upon him), "Nay, (We follow) only the religion of Ibrahim (Abraham), Hanifa [Islamic Monotheism, i.e. to worship none but Allah (Alone)], and he was not of Al-Mushrikun (those who worshipped others along with Allah - see V. 2:105)."
  UR: کہا اے میرے رب تو میری مدد کر کیوں کہ انہو ں نے مجھے جھٹلایا ہے  →  EN: [Noah] said, "My Lord, support me because they have denied me

In [ ]:
len(all_pairs)

3000

---
## Section 2 — Tokenization

We need to split sentences into word tokens before we can work with them.

### ✏️ Task 2.1 — Complete the `tokenize` function

- **English:** lowercase the text, then extract only word characters using `re.findall(r"\w+", ...)`
- **Urdu:** Urdu uses Arabic script — just split on whitespace with `.split()`

In [ ]:
def tokenize(sentence, lang):
    """
    Split a sentence into a list of word tokens.

    Parameters:  sentence (str), lang ('en' or 'ur')
    Returns:     list of str tokens

    Examples:
        tokenize('I am Fine!', 'en')       →  ['i', 'am', 'fine']
        tokenize('میں ٹھیک ہوں', 'ur')    →  ['میں', 'ٹھیک', 'ہوں']
    """
    # ✏️ YOUR CODE HERE
    pass


# --- Tests (do not modify) ---
assert tokenize('I am Fine!', 'en') == ['i', 'am', 'fine']
assert tokenize('میں ٹھیک ہوں', 'ur') == ['میں', 'ٹھیک', 'ہوں']
print('✅ tokenize() tests passed!')

---
## Section 3 — Vocabulary

An RNN cannot read text — it needs numbers. A **vocabulary** maps every word to a unique integer ID.

We reserve four special token IDs:

| Token | Meaning | ID |
|-------|---------|----|
| `<PAD>` | Padding | 0 |
| `<SOS>` | Start of sentence | 1 |
| `<EOS>` | End of sentence | 2 |
| `<UNK>` | Unknown word | 3 |

In [ ]:
# ✅ PROVIDED
PAD_IDX, SOS_IDX, EOS_IDX, UNK_IDX = 0, 1, 2, 3

class Vocabulary:
    def __init__(self, lang):
        self.lang = lang
        self.word2idx = {'<PAD>': 0, '<SOS>': 1, '<EOS>': 2, '<UNK>': 3}
        self.idx2word = {0: '<PAD>', 1: '<SOS>', 2: '<EOS>', 3: '<UNK>'}
        self._counts  = Counter()

    def add_sentence(self, sentence):
        for word in tokenize(sentence, self.lang):
            self._counts[word] += 1

    def build(self, min_freq=1):
        for word, count in self._counts.items():
            if count >= min_freq and word not in self.word2idx:
                idx = len(self.word2idx)
                self.word2idx[word] = idx
                self.idx2word[idx]  = word

    def encode(self, sentence):
        """sentence string  →  list of int IDs"""
        return [self.word2idx.get(w, UNK_IDX) for w in tokenize(sentence, self.lang)]

    def decode(self, indices):
        """list of int IDs  →  sentence string"""
        words = []
        for i in indices:
            if i in (SOS_IDX, PAD_IDX): continue
            if i == EOS_IDX: break
            words.append(self.idx2word.get(i, '<UNK>'))
        return ' '.join(words)

    def __len__(self):
        return len(self.word2idx)

### ✏️ Task 3.1 — Build both vocabularies

1. Create `src_vocab = Vocabulary('ur')` and `trg_vocab = Vocabulary('en')`.
2. Loop over `all_pairs` and call `.add_sentence()` on the correct vocab for each language.
3. Call `.build(min_freq=1)` on both.
4. Print the vocabulary sizes.

In [ ]:
# ✏️ YOUR CODE HERE
src_vocab = None   # Vocabulary('ur')
trg_vocab = None   # Vocabulary('en')

# Loop and add sentences ...

# Build ...

# Print sizes ...
print(f'Urdu vocab size   : {len(src_vocab)}')
print(f'English vocab size: {len(trg_vocab)}')

---
## Section 4 — Dataset & DataLoader

In [ ]:
# ✅ PROVIDED — train/val split
random.shuffle(all_pairs)
split      = int(len(all_pairs) * 0.9)
train_pairs = all_pairs[:split]
val_pairs   = all_pairs[split:]
print(f'Train: {len(train_pairs)}  |  Val: {len(val_pairs)}')

### ✏️ Task 4.1 — Complete `TranslationDataset`

Implement `__len__` and `__getitem__`.

`__getitem__` should:
1. Get the `(urdu, english)` pair at position `idx`.
2. Encode the Urdu sentence and wrap it: `[SOS_IDX] + encoded_tokens + [EOS_IDX]`.
3. Do the same for English.
4. Return both as `torch.LongTensor`.

In [ ]:
class TranslationDataset(Dataset):
    def __init__(self, pairs, src_vocab, trg_vocab):
        self.pairs     = pairs
        self.src_vocab = src_vocab
        self.trg_vocab = trg_vocab

    def __len__(self):
        # ✏️ YOUR CODE HERE
        pass

    def __getitem__(self, idx):
        """
        Return
        ------
        src_tensor : LongTensor  [SOS, w1, w2, ..., EOS]
        trg_tensor : LongTensor  [SOS, w1, w2, ..., EOS]
        """
        # ✏️ YOUR CODE HERE
        pass


# ✅ PROVIDED — collate & loaders
def collate_fn(batch):
    src_batch, trg_batch = zip(*batch)
    src_pad = pad_sequence(src_batch, batch_first=False, padding_value=PAD_IDX)
    trg_pad = pad_sequence(trg_batch, batch_first=False, padding_value=PAD_IDX)
    return src_pad, trg_pad

BATCH_SIZE   = 32
train_loader = DataLoader(TranslationDataset(train_pairs, src_vocab, trg_vocab),
                          batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn)
val_loader   = DataLoader(TranslationDataset(val_pairs,   src_vocab, trg_vocab),
                          batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

src_sample, trg_sample = next(iter(train_loader))
print(f'Source batch shape: {src_sample.shape}   (seq_len × batch_size)')
print(f'Target batch shape: {trg_sample.shape}')

---
## Section 5 — RNN Model

We use **vanilla RNN** (`nn.RNN`) — exactly what you studied in class.

Recall the RNN update rule at each time step *t*:

$$h_t = \tanh(W_{ih} \cdot x_t + W_{hh} \cdot h_{t-1} + b)$$

The **Encoder** reads the whole Urdu sequence and produces a final hidden state.  
The **Decoder** uses that hidden state to generate English words one at a time.

### ✏️ Task 5.1 — Implement the Encoder

The Encoder needs three components:
- `nn.Embedding(src_vocab_size, emb_dim)` — turns token IDs into dense vectors
- `nn.Dropout(dropout)` — regularisation
- `nn.RNN(emb_dim, hidden_dim, num_layers, batch_first=False)` — the RNN itself

In `forward`, pass `src` through embedding → dropout → RNN, then **return only `hidden`**  
(we don't need all intermediate outputs for the basic model).

In [ ]:
class Encoder(nn.Module):
    """
    Reads the full Urdu sequence and compresses it into a hidden state vector.

    forward(src)  →  hidden
      src    : LongTensor  [src_len, batch]
      hidden : FloatTensor [num_layers, batch, hidden_dim]
    """
    def __init__(self, src_vocab_size, emb_dim, hidden_dim, num_layers, dropout):
        super().__init__()
        # ✏️ YOUR CODE HERE
        # Define self.embedding, self.dropout, self.rnn  (use nn.RNN)
        pass

    def forward(self, src):
        # ✏️ YOUR CODE HERE
        # 1. embedded = dropout(embedding(src))   →  [src_len, batch, emb_dim]
        # 2. _, hidden = rnn(embedded)            →  hidden: [num_layers, batch, hidden_dim]
        # 3. return hidden
        pass

### ✏️ Task 5.2 — Implement the Decoder

The Decoder runs **one step at a time** — it takes a single token and the current hidden state, and produces the next token's scores.

Components needed:
- `nn.Embedding(trg_vocab_size, emb_dim)`
- `nn.Dropout(dropout)`
- `nn.RNN(emb_dim, hidden_dim, num_layers, batch_first=False)`
- `nn.Linear(hidden_dim, trg_vocab_size)` — projects hidden state to vocabulary scores

In `forward`:
1. `token` arrives as shape `[batch]` — unsqueeze it to `[1, batch]` so the RNN sees it as one time step.
2. Embed + dropout → `[1, batch, emb_dim]`
3. One RNN step → `output [1, batch, hidden_dim]`, new `hidden`
4. Squeeze output → `[batch, hidden_dim]`
5. Project with linear → `prediction [batch, trg_vocab_size]`
6. Return `prediction, hidden`

In [ ]:
class Decoder(nn.Module):
    """
    Generates one English token per call.

    forward(token, hidden)  →  prediction, hidden
      token      : LongTensor   [batch]
      hidden     : FloatTensor  [num_layers, batch, hidden_dim]
      prediction : FloatTensor  [batch, trg_vocab_size]  (raw scores / logits)
    """
    def __init__(self, trg_vocab_size, emb_dim, hidden_dim, num_layers, dropout):
        super().__init__()
        # ✏️ YOUR CODE HERE
        # Define self.embedding, self.dropout, self.rnn (nn.RNN), self.fc_out (nn.Linear)
        pass

    def forward(self, token, hidden):
        # ✏️ YOUR CODE HERE
        pass

### ✏️ Task 5.3 — Implement the Seq2Seq wrapper

This ties the Encoder and Decoder together.

In `forward`:
1. Encode `src` → `hidden`.
2. The first decoder input is the `<SOS>` token: `token = trg[0, :]`.
3. Loop from `t = 1` to `trg_len - 1`:
   - Call `decoder(token, hidden)` → `prediction, hidden`
   - Store `prediction` in `outputs[t]`
   - **Teacher forcing:** with probability `teacher_forcing_ratio`, use the real next word `trg[t]` as the next input; otherwise use `prediction.argmax(dim=1)`.
4. Return `outputs`.

> **What is teacher forcing?** During training, instead of always feeding the model's own (possibly wrong) prediction back as input, we sometimes feed the correct word. This makes training more stable.

In [ ]:
class Seq2Seq(nn.Module):
    """
    Full translation model.

    forward(src, trg)  →  outputs
      src     : LongTensor   [src_len, batch]
      trg     : LongTensor   [trg_len, batch]
      outputs : FloatTensor  [trg_len, batch, trg_vocab_size]
    """
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device  = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        trg_len        = trg.shape[0]
        batch_size     = trg.shape[1]
        trg_vocab_size = self.decoder.fc_out.out_features

        # Storage tensor for all decoder outputs
        outputs = torch.zeros(trg_len, batch_size, trg_vocab_size).to(self.device)

        # ✏️ YOUR CODE HERE
        # Step 1: hidden = self.encoder(src)
        # Step 2: token  = trg[0, :]   (first input = <SOS>)
        # Step 3: loop t from 1 to trg_len-1
        #           prediction, hidden = self.decoder(token, hidden)
        #           outputs[t] = prediction
        #           teacher_force = random.random() < teacher_forcing_ratio
        #           token = trg[t] if teacher_force else prediction.argmax(dim=1)

        return outputs

### Assemble the model

In [ ]:
# ✅ PROVIDED — hyperparameters
EMB_DIM    = 128
HIDDEN_DIM = 256
NUM_LAYERS = 1
DROPOUT    = 0.3

encoder = Encoder(len(src_vocab), EMB_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT).to(DEVICE)
decoder = Decoder(len(trg_vocab), EMB_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT).to(DEVICE)
model   = Seq2Seq(encoder, decoder, DEVICE).to(DEVICE)

total = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total trainable parameters: {total:,}')

---
## Section 6 — Training

### ✏️ Task 6.1 — Implement `train_epoch`

For each batch:
1. Move `src` and `trg` to `DEVICE`.
2. Zero the gradients (`optimizer.zero_grad()`).
3. Forward pass: `output = model(src, trg)`.
4. Compute loss — reshape carefully:
   - Skip index 0 (the `<SOS>` token at position 0)
   - `output` shape `[trg_len, batch, vocab]` → `output[1:].reshape(-1, vocab_size)`
   - `trg`    shape `[trg_len, batch]`        → `trg[1:].reshape(-1)`
5. `loss.backward()`
6. Clip gradients: `nn.utils.clip_grad_norm_(model.parameters(), 1.0)`
7. `optimizer.step()`

Return the average loss across all batches.

In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
optimizer = optim.Adam(model.parameters(), lr=1e-3)


def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0

    for src, trg in loader:
        # ✏️ YOUR CODE HERE
        pass

    return total_loss / len(loader)

### ✏️ Task 6.2 — Implement `evaluate`

Same as `train_epoch` but:
- Set model to **eval mode**.
- Wrap everything in `torch.no_grad()` — skip gradient computation.
- Pass `teacher_forcing_ratio=0` so the model uses its own predictions.

In [ ]:
def evaluate(model, loader, criterion):
    # ✏️ YOUR CODE HERE
    pass

### Run the training loop

In [ ]:
# ✅ PROVIDED
N_EPOCHS      = 10
best_val_loss = float('inf')
train_losses, val_losses = [], []

for epoch in range(1, N_EPOCHS + 1):
    t0         = time.time()
    train_loss = train_epoch(model, train_loader, optimizer, criterion)
    val_loss   = evaluate(model, val_loader, criterion)
    elapsed    = time.time() - t0

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    flag = ''
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_model.pt')
        flag = '  ← best'

    print(f'Epoch {epoch:02d} ({elapsed:.1f}s)  '
          f'Train Loss: {train_loss:.3f}  Val Loss: {val_loss:.3f}{flag}')

### ✏️ Task 6.3 — Plot the loss curves

Plot `train_losses` and `val_losses` against epoch number on the same graph.  
Add a legend, axis labels, and a title.

In [ ]:
# ✏️ YOUR CODE HERE
plt.figure(figsize=(8, 4))

# plot both curves ...

plt.tight_layout()
plt.show()

---
## Section 7 — Translate & Evaluate

### ✏️ Task 7.1 — Implement `translate`

Translate one Urdu sentence at inference time (no teacher forcing):

1. Encode the sentence: `[SOS] + src_vocab.encode(sentence) + [EOS]` → `LongTensor` → move to device.
2. Pass through encoder → `hidden`.
3. Start with `token = tensor([SOS_IDX])` on device.
4. Loop up to `max_len=50` steps:
   - `prediction, hidden = decoder(token, hidden)`
   - `next_token = prediction.argmax(dim=1)`
   - If `next_token == EOS_IDX`: stop.
   - Otherwise append `next_token.item()` to a list, set `token = next_token`.
5. Return `trg_vocab.decode(collected_ids)`.

In [ ]:
def translate(sentence, src_vocab, trg_vocab, model, device, max_len=50):
    """
    Translate a single Urdu sentence to English.
    Returns a string.
    """
    model.eval()
    with torch.no_grad():
        # ✏️ YOUR CODE HERE
        pass


# ✅ PROVIDED — test a few samples
model.load_state_dict(torch.load('best_model.pt', map_location=DEVICE))

print('--- Sample Translations ---\n')
for ur, ref in random.sample(val_pairs, min(5, len(val_pairs))):
    pred = translate(ur, src_vocab, trg_vocab, model, DEVICE)
    print(f'Urdu     : {ur}')
    print(f'Reference: {ref}')
    print(f'Predicted: {pred}\n')

### ✏️ Task 7.2 — Compute BLEU Score

**BLEU** measures how much your predicted translations overlap with the reference translations (0–100, higher is better).

1. Loop over all `val_pairs`, call `translate()` for each.
2. Collect predictions in `hypotheses` (list of str) and references in `references` (list of str).
3. Call `sacrebleu.corpus_bleu(hypotheses, [references])` and print the score.

In [ ]:
# ✏️ YOUR CODE HERE
hypotheses = []
references  = []

for ur, ref in val_pairs:
    pass  # translate and append

bleu = sacrebleu.corpus_bleu(hypotheses, [references])
print(f'BLEU score: {bleu.score:.2f}')

---
## Section 8 — Reflection Questions

**Answer in this cell (double-click to edit):**

**Q1.** What does the encoder's final hidden state represent? What information might be lost when the source sentence is very long?

*Your answer...*

---

**Q2.** What is teacher forcing? What could go wrong if you train with `teacher_forcing_ratio = 1.0` and then test with `ratio = 0`?

*Your answer...*

---

**Q3.** We used `nn.RNN`. What is one weakness of a vanilla RNN compared to an LSTM or GRU, and why does it matter for translation?

*Your answer...*

---

**Q4.** Look at your worst-performing translations. What patterns do you notice? Why do you think the model struggles with those examples?

*Your answer...*

---
## ✅ Submission Checklist

- [ ] Task 2.1 — `tokenize()` passes both assertions
- [ ] Task 3.1 — Both vocabularies built and sizes printed
- [ ] Task 4.1 — `TranslationDataset` fully implemented
- [ ] Task 5.1 — `Encoder` implemented with `nn.RNN`
- [ ] Task 5.2 — `Decoder` implemented with `nn.RNN`
- [ ] Task 5.3 — `Seq2Seq.forward()` with teacher forcing
- [ ] Task 6.1 — `train_epoch` implemented
- [ ] Task 6.2 — `evaluate` implemented
- [ ] Task 6.3 — Loss curves plotted
- [ ] Task 7.1 — `translate` implemented and tested
- [ ] Task 7.2 — BLEU score computed
- [ ] Task 8   — All reflection questions answered
- [ ] All cells run top-to-bottom without errors